## Resumen del EDA: ¿qué pedidos llegan tarde?

**Qué hice:** analice 96.470 pedidos entregados de Olist para ver que factores se asocian con que un pedido llegue despues del día prometido (`tarde_dia`). En total llega tarde el 6,77 % (IC 95 %: 6,62 % a 6,93 % esto es el intervalo entre que valores puede estar el porcentaje real de retrasos.).

**Cómo lo hice:** calcule el porcentaje de pedidos tarde por grupo (estado, categoria, dias prometidos, peso y flete) y le agregue a cada porcentaje un intervalo de confianza del 95 % (método de Wilson). Asi separo las diferencias reales de las que pueden ser casualidad, sobre todo en los grupos con pocos pedidos. En las categorias use un minimo de 100 pedidos.

**Qué encontré:**
- **Estado:** hay asociación real pero modesta entre estado del cliente y retraso (chi-cuadrado, V de Cramér ≈ 0,14). El nordeste tiene cerca del 21 % de pedidos tarde, contra alrededor del 4,5 % en otros estados.
- **Mismo estado:** si vendedor y cliente están en el mismo estado, el 4,50 % llega tarde; en estados distintos, el 8,05 %.
- **Categoría:** audio (11,9 %) y bebes (8,2 %) están claramente por encima del promedio; utilidades_domesticas, pet_shop, eletrodomesticos y malas_acessorios, por debajo. En la mayoría de las categorías no hay diferencia clara por tener pocos pedidos.
- **Días prometidos:** la relación no es lineal. El retraso sube hasta el tramo de 19 a 23 días (8,3 %) y baja después (5,1 % en los más largos).
- **Flete:** los pedidos con flete más barato (hasta 13,92) llegan tarde menos veces (4,4 %) que el resto (7,0 % a 8,0 %).
- **Peso:** el efecto es pequeño; solo los pedidos más pesados (más de 2.050 g) se separan, con un 7,4 %.

**Decisiones sobre los datos:** los nulos de categoría los dejé como `sin_categoria` (1,4 % de los pedidos). Los 22 pedidos con peso 0 se tratan como datos faltantes, y los 336 con flete 0 se conservan.

In [4]:
# Cargo los datos de pedidos
import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv("../data/processed/pedidos_unidos.csv")

print(df.head())

cols_fecha = ["order_purchase_timestamp",	"order_approved_at",	"order_delivered_carrier_date",	"order_delivered_customer_date",	"order_estimated_delivery_date"]


                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [5]:
# Convierto las columnas de fecha a datetime (de str a datetime) para poder leer mejor.

for pedidos in cols_fecha:
    df[pedidos] = pd.to_datetime(df[pedidos])

df.info()


    

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_state                 99441 non-null  str           
 9   customer_city                  99441 non-null  str           
 10  n_items                        98666 non-null  float64       
 11  precio_total              

In [6]:
# Paso a filtrar por pedidos entregados. 

print(df.value_counts("order_status")) # Me fijo cuantos entregados hay.
mask = df["order_status"] == "delivered"
mask.sum()
df_ent = df[mask]
print(df_ent.shape)
print(df_ent.isna().sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
(96478, 17)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
customer_state                      0
customer_city                       0
n_items                             0
precio_total                        0
flete_total                         0
n_vendedores                        0
peso_total_g                        0
categoria                        1332
estado_vendedor                     0
dtype: int64


In [7]:
# Paso a eliminar las 8 filas con nulos en order_delivered_customer_date

df_ent = df_ent.dropna(subset=["order_delivered_customer_date"]).copy()
print(df_ent)



                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status order_purchase_timestamp   order_approved_at  \
0       

In [8]:
# Creo una columna nueva llamada "tarde" para comparar los pedidos que llegaron tarde. 
df_ent["tarde"] = (df_ent["order_delivered_customer_date"] > df_ent["order_estimated_delivery_date"]).astype(int)
df_ent["tarde"].value_counts(normalize = True) # casi 91% de pedidos a tiempo, lo cual me daria un modelo que siempre diria "a tiempo". Target desbalanceado. 


tarde
0    0.918876
1    0.081124
Name: proportion, dtype: float64

In [9]:
# Aca comparo SOLO EL DIA de entrega, SIN LA HORA.
df_ent["tarde_dia"] = (df_ent["order_delivered_customer_date"].dt.normalize() > df_ent["order_estimated_delivery_date"]).astype(int) # normalize deja la hora real en 00:00:00
print(df_ent["tarde"].mean() * 100)
print(df_ent["tarde_dia"].mean() * 100)

# Uso tarde_dia como target ya que, tecnicamente, se cumple con el criterio de que llego el dia X. 
# Con la definición con hora, tendrías aprox. 1.290 falsas alarmas. 
# Un modelo entrenado así avisaría de retrasos que no son tales, y el equipo de logística perdería confianza en él.

8.112366538820359
6.7730900798175595


In [10]:
# Primer EDA

df_ent.groupby("customer_state")["tarde_dia"].mean().sort_values(ascending = False) # Porcentaje por estado en Brasil


customer_state
AL    0.214106
MA    0.174338
SE    0.152239
PI    0.138655
CE    0.137608
RR    0.121951
BA    0.121622
RJ    0.121053
PA    0.112051
ES    0.107268
PB    0.104449
TO    0.098540
MS    0.097004
PE    0.096045
RN    0.092827
SC    0.082064
GO    0.065406
RS    0.060816
MT    0.059819
DF    0.056731
MG    0.045711
SP    0.044945
PR    0.040423
AC    0.037500
AP    0.029851
RO    0.028807
AM    0.027586
Name: tarde_dia, dtype: float64

In [11]:
# Pedidos por estados para saber que porcentajes de retraso son fiables (estados con pocos pedidos no se interpretan)
df_ent.groupby("customer_state").size().sort_values(ascending=False)

customer_state
SP    40494
RJ    12350
MG    11354
RS     5344
PR     4923
SC     3546
BA     3256
DF     2080
ES     1995
GO     1957
PE     1593
CE     1279
PA      946
MT      886
MA      717
MS      701
PB      517
PI      476
RN      474
AL      397
SE      335
TO      274
RO      243
AM      145
AC       80
AP       67
RR       41
dtype: int64

In [12]:
# Armar tabla con crosstab y chi2

tabla_contingencia = pd.crosstab(df_ent["customer_state"], df_ent["tarde_dia"])

chi2, p, grados, esperada = chi2_contingency(tabla_contingencia)

print(chi2)
print(p)
print(grados)

print((esperada < 5).sum())

# Test de chi2 entre customer_state y tarde_dia donde el retraso depende del estado pero la asociacion es modesta por lo que tengo que combinarlo con otras variables.



1785.7927506269548
0.0
26
2


In [13]:
# Voy a mirar cuales son las categorias que tienen nulos

print(df_ent["categoria"].isnull().sum())
print(df_ent["categoria"].isnull().any())
print(df_ent[df_ent["categoria"].isnull()])

# 1332 pedidos NO TIENEN CATEGORIA, por ende aparecen como nulos a pesar de estar entregados. 

1332
True
                               order_id                       customer_id  \
107    bfe42c22ecbf90bc9f35cf591270b6a7  803ac05904124294f8767894d6da532b   
180    58ac1947c1a9067b9f416cba6d844a3f  ee8e1d37f563ecc11cc4dcb4dfd794c2   
228    e22b71f6e4a481445ec4527cb4c405f7  1faf89c8f142db3fca6cf314c51a37b6   
263    a094215e786240fcfefb83d18036a1cd  86acfb656743da0c113d176832c9d535   
331    6f841dde94727854eaff3f66432c80ea  a9c9532060c9d245f06526c633d2dfba   
...                                 ...                               ...   
99069  1a10e938a1c7d8e5eecc3380f71ca76b  8a81607347c25d881d995d94de6ad824   
99215  e33865519137f5737444109ae8438633  64b086bdcc54458af3ea3bd838db54a5   
99222  f0dd9af88d8ef5a8e4670fbbedaf19c4  30ddb50bd22ee927ebe308ea3da60735   
99228  272874573723eec18f23c0471927d778  48e080c8001e92ebea2b64e474f91a60   
99245  dff2b9b8d7cfc595836945e1443789c3  2436fb2666a65fbacae82532e797cabf   

      order_status order_purchase_timestamp   order_approved_at  

In [14]:
# Reetiquetar los nulos ya que no los voy a eliminar al ser pocos.
df_ent["categoria"] = df_ent["categoria"].fillna("sin_categoria")
print(df_ent["categoria"].isnull().sum())   # deberia dar 0

# Conteo y % de tarde por categoría
tabla_cat = (
    df_ent.groupby("categoria")["tarde_dia"]
    .agg(pedidos="count", pct_tarde="mean") # calculo dos cosas, media y la proporcion.
    .reset_index()
)

# Filtrar categorias con pocos pedidos y ordenar
tabla_cat = tabla_cat[tabla_cat["pedidos"] >= 100]
tabla_cat["pct_tarde"] = (tabla_cat["pct_tarde"] * 100).round(1)
tabla_cat = tabla_cat.sort_values("pct_tarde", ascending=False)

tabla_cat.head(10)

0


,categoria,pedidos,pct_tarde
7,audio,344,11.9
14,casa_conforto,370,9.5
39,fashion_underwear_e_moda_praia,117,9.4
49,livros_tecnicos,255,8.2
9,bebes,2763,8.2
55,moveis_escritorio,1246,8.1
6,artigos_de_natal,125,8.0
30,eletronicos,2507,7.7
57,moveis_sala,409,7.6
45,instrumentos_musicais,608,7.6


In [15]:
tabla_cat[tabla_cat["categoria"] == "sin_categoria"]

,categoria,pedidos,pct_tarde
68,sin_categoria,1332,7.1


In [16]:
# Feature 1: mismo_estado es para ver si vendedor y cliente estan en el mismo estado o no ya que asi recorreria menos distancia.
df_ent["mismo_estado"] = (df_ent["customer_state"] == df_ent["estado_vendedor"]).astype(int)
print(df_ent.groupby("mismo_estado")["tarde_dia"].mean().sort_values(ascending = False))
df_ent["mismo_estado"].value_counts()


mismo_estado
0    0.080477
1    0.045042
Name: tarde_dia, dtype: float64


mismo_estado
0    61769
1    34701
Name: count, dtype: int64

In [17]:
df_ent[["order_estimated_delivery_date"]].dtypes 
df_ent["dias_prometidos"] = (df_ent["order_estimated_delivery_date"] - df_ent["order_purchase_timestamp"]).dt.days

df_ent.describe()



,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,precio_total,flete_total,n_vendedores,peso_total_g,tarde,tarde_dia,mismo_estado,dias_prometidos
count,96470,96456,96469,96470,96470,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000
mean,2018-01-01 23:17:43.624411,2018-01-02 10:41:38.246330,2018-01-05 04:46:44.911992,2018-01-14 12:41:33.581683,2018-01-25 16:58:03.644656,1.142210,137.040001,22.785798,1.013901,2386.472437,0.081124,0.067731,0.359708,23.372748
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,0.850000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,2017-09-14 08:56:46.500000,2017-09-14 14:43:49,2017-09-18 16:42:10,2017-09-25 22:15:09.500000,2017-10-05 00:00:00,1.000000,45.900000,13.850000,1.000000,300.000000,0.000000,0.000000,0.000000,18.000000
50%,2018-01-20 19:34:43.500000,2018-01-22 13:49:23.500000,2018-01-24 16:03:24,2018-02-02 19:32:21,2018-02-16 00:00:00,1.000000,86.500000,17.170000,1.000000,750.000000,0.000000,0.000000,0.000000,23.000000
75%,2018-05-05 18:29:50.250000,2018-05-06 10:52:55.750000,2018-05-08 14:32:00,2018-05-15 22:54:48.500000,2018-05-28 00:00:00,1.000000,149.900000,24.020000,1.000000,2050.000000,0.000000,0.000000,1.000000,28.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,21.000000,13440.000000,1794.960000,5.000000,184400.000000,1.000000,1.000000,1.000000,155.000000
std,NaN,NaN,NaN,NaN,NaN,0.538824,209.052608,21.559959,0.123542,4769.813318,0.273026,0.251285,0.479917,8.758421


In [18]:
df_ent["tramo_plazo"] = pd.qcut(df_ent["dias_prometidos"], q=4) # parte los pedidos en 4 grupos de igual tamanio.

tabla_plazo = (
    df_ent.groupby("tramo_plazo", observed=True)["tarde_dia"] # observed=true evita un aviso de pandas con columnas de tipo categorico.
    .agg(pedidos="count", pct_tarde="mean")
    .reset_index()
)
tabla_plazo["pct_tarde"] = (tabla_plazo["pct_tarde"] * 100).round(1)

tabla_plazo

,tramo_plazo,pedidos,pct_tarde
0,"(1.999, 18.0]",26000,6.3
1,"(18.0, 23.0]",25933,8.3
2,"(23.0, 28.0]",21871,7.3
3,"(28.0, 155.0]",22666,5.1


In [20]:
from statsmodels.stats.proportion import proportion_confint # calcula el intervalo de confianza de una proporcion

def ic_tarde(df, columna, min_pedidos=1):
    """Porcentaje de pedidos tarde por grupo, con IC 95% de Wilson."""
    g = df.groupby(columna, observed=True)["tarde_dia"].agg(tarde="sum", n="count")
    g = g[g["n"] >= min_pedidos].copy()
    low, high = proportion_confint(g["tarde"], g["n"], alpha=0.05, method="wilson") # alpha es el intervalo de confianza del 95%
    g["pct_tarde"] = (g["tarde"] / g["n"] * 100).round(2)
    g["ic_inf"] = (low * 100).round(2)
    g["ic_sup"] = (high * 100).round(2)
    return g.sort_values("pct_tarde", ascending=False)

# Referencia global
tarde_total = int(df_ent["tarde_dia"].sum())
n_total = len(df_ent)
low, high = proportion_confint(tarde_total, n_total, alpha=0.05, method="wilson")
print(f"Global: {tarde_total / n_total * 100:.2f}%  IC95: [{low*100:.2f}%, {high*100:.2f}%]")

# Aca abajo empezamos a aplicar todo


# 1) mismo_estado
print(ic_tarde(df_ent, "mismo_estado"))

# 2) categoría (mínimo 100 pedidos)
print(ic_tarde(df_ent, "categoria", min_pedidos=100))

# 3) cuartiles de dias_prometidos
df_ent["q_dias"] = pd.qcut(df_ent["dias_prometidos"], 4)
print(ic_tarde(df_ent, "q_dias").sort_index())

Global: 6.77%  IC95: [6.62%, 6.93%]
              tarde      n  pct_tarde  ic_inf  ic_sup
mismo_estado                                         
0              4971  61769       8.05    7.84    8.26
1              1563  34701       4.50    4.29    4.73
                                                tarde     n  pct_tarde  \
categoria                                                                
audio                                              41   344      11.92   
casa_conforto                                      35   370       9.46   
fashion_underwear_e_moda_praia                     11   117       9.40   
livros_tecnicos                                    21   255       8.24   
bebes                                             226  2763       8.18   
moveis_escritorio                                 101  1246       8.11   
artigos_de_natal                                   10   125       8.00   
eletronicos                                       192  2507       7.66   
moveis_s

In [21]:
# Revisar los ceros en los pesos
for col in ["peso_total_g", "flete_total"]:
    ceros = (df_ent[col] == 0).sum()
    print(f"{col}: {ceros} ceros ({ceros / len(df_ent) * 100:.2f}%)")
    print(df_ent[col].describe().round(2), "\n")
# Cuantos pedidos tienen 0 y como se reparte el resto (min, mediana, máx)

# Peso: grupo aparte para los ceros + cuartiles del resto
df_ent["peso_grupo"] = "peso_0"
mask_peso = df_ent["peso_total_g"] > 0
df_ent.loc[mask_peso, "peso_grupo"] = pd.qcut(
    df_ent.loc[mask_peso, "peso_total_g"], 4, duplicates="drop"
).astype(str)
# Los pedidos con peso 0 van a su propio grupo; los demas se dividen en 4 cuartiles
# .astype(str) evita problemas al mezclar texto con intervalos
print(ic_tarde(df_ent, "peso_grupo").sort_index())

#  Flete: igual, ceros aparte + cuartiles del resto
df_ent["flete_grupo"] = "flete_0"
mask_flete = df_ent["flete_total"] > 0
df_ent.loc[mask_flete, "flete_grupo"] = pd.qcut(
    df_ent.loc[mask_flete, "flete_total"], 4, duplicates="drop"
).astype(str)
print(ic_tarde(df_ent, "flete_grupo").sort_index())

peso_total_g: 22 ceros (0.02%)
count     96470.00
mean       2386.47
std        4769.81
min           0.00
25%         300.00
50%         750.00
75%        2050.00
max      184400.00
Name: peso_total_g, dtype: float64 

flete_total: 336 ceros (0.35%)
count    96470.00
mean        22.79
std         21.56
min          0.00
25%         13.85
50%         17.17
75%         24.02
max       1794.96
Name: flete_total, dtype: float64 

                    tarde      n  pct_tarde  ic_inf  ic_sup
peso_grupo                                                 
(1.999, 300.0]       1567  25199       6.22    5.93    6.52
(2050.0, 184400.0]   1791  24088       7.44    7.11    7.77
(300.0, 750.0]       1584  23577       6.72    6.41    7.05
(750.0, 2050.0]      1592  23584       6.75    6.44    7.08
peso_0                  0     22       0.00    0.00   14.87
                  tarde      n  pct_tarde  ic_inf  ic_sup
flete_grupo                                              
(13.92, 17.21]     1686  23969   

In [22]:
df_ent.to_csv("../data/processed/pedidos_modelo.csv", index=False)
# guardo el CSV ya limpio.
# index=False evita que se guarde el indice como columna extra